In [1]:
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git

Cloning into 'SentimentAnalysis'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 62 (delta 17), reused 52 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 87.81 KiB | 946.00 KiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
%cd /content/SentimentAnalysis
!git pull origin main

/content/SentimentAnalysis
From https://github.com/Chetnapadhi/SentimentAnalysis
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [5]:
%cd /content/SentimentAnalysis
!pip install -q -r requirements.txt

/content/SentimentAnalysis
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 83.6 MB/s eta 0:00:00


In [6]:
import torch
import transformers
import sklearn

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

Torch: 2.11.0+cu128
Transformers: 5.16.1
CUDA: True


In [7]:
!ls -lh \
notebooks/05_e2_pretrained_emoji.ipynb \
src/train_e2.py \
src/train_e2_pretrain.py \
run_e2.py

-rw-r--r-- 1 root root  35K Sep  7 19:09 notebooks/05_e2_pretrained_emoji.ipynb
-rw-r--r-- 1 root root  934 Sep  7 19:09 run_e2.py
-rw-r--r-- 1 root root 8.2K Sep  7 19:09 src/train_e2_pretrain.py
-rw-r--r-- 1 root root  12K Sep  7 19:09 src/train_e2.py


In [8]:
!find src -iname "*e2*" -o -iname "*emoji*"

src/embeddings/emoji_encoder.py
src/train_e2.py
src/train_e2_pretrain.py
src/models/e2_concat_fusion.py


In [9]:
# CELL 6A — Check canonical dataset files

import os

base = "/content/SentimentAnalysis/data/processed/canonical"

print("Current directory:")
!pwd

print("\nCanonical directory exists:", os.path.exists(base))

if os.path.exists(base):
    print("\nFiles in canonical directory:")
    !ls -lah /content/SentimentAnalysis/data/processed/canonical/
else:
    print("❌ Canonical directory does not exist")

Current directory:
/content/SentimentAnalysis

Canonical directory exists: False
❌ Canonical directory does not exist


In [10]:
# CELL 6B — Check file sizes

import os

files = [
    "/content/SentimentAnalysis/data/processed/canonical/final_train.jsonl",
    "/content/SentimentAnalysis/data/processed/canonical/final_validation.jsonl",
    "/content/SentimentAnalysis/data/processed/canonical/final_test.jsonl",
]

for f in files:
    print("\n", f)

    if os.path.exists(f):
        print("  Exists: YES")
        print("  Size:", os.path.getsize(f), "bytes")

        if os.path.getsize(f) > 0:
            with open(f, "r", encoding="utf-8") as file:
                first_line = file.readline()

            print("  First line:")
            print(first_line[:500])
        else:
            print("  ❌ FILE IS EMPTY")
    else:
        print("  ❌ FILE DOES NOT EXIST")


 /content/SentimentAnalysis/data/processed/canonical/final_train.jsonl
  ❌ FILE DOES NOT EXIST

 /content/SentimentAnalysis/data/processed/canonical/final_validation.jsonl
  ❌ FILE DOES NOT EXIST

 /content/SentimentAnalysis/data/processed/canonical/final_test.jsonl
  ❌ FILE DOES NOT EXIST


In [11]:
# CELL 6C — Find dataset preparation scripts

import os

repo = "/content/SentimentAnalysis"

print("Possible data/preprocessing files:\n")

for root, dirs, files in os.walk(repo):
    # Skip unnecessary folders
    dirs[:] = [
        d for d in dirs
        if d not in [".git", ".venv", "venv", "__pycache__", ".ipynb_checkpoints"]
    ]

    for file in files:
        if file.endswith((".py", ".ipynb")):
            path = os.path.join(root, file)

            # Look for files likely related to data preparation
            name = file.lower()
            if any(x in name for x in [
                "data", "prep", "process", "canonical",
                "dataset", "split", "stocktwits"
            ]):
                print(path)


Possible data/preprocessing files:

/content/SentimentAnalysis/src/data/validate_data.py
/content/SentimentAnalysis/src/data/stocktwits_adapter.py
/content/SentimentAnalysis/src/data/build_final_dataset.py
/content/SentimentAnalysis/src/data/inspect_datasets.py
/content/SentimentAnalysis/src/data/preprocessing.py


In [12]:
# CELL 6D — Check the data directories

!find /content/SentimentAnalysis/data -maxdepth 4 -type f | sort

/content/SentimentAnalysis/data/validation/findings_report.json
/content/SentimentAnalysis/data/validation/integrity_report.json


In [13]:
# CELL 6D — Verify canonical dataset

import os
import pandas as pd

base = "/content/SentimentAnalysis/data/processed/canonical"

files = {
    "train": f"{base}/final_train.jsonl",
    "validation": f"{base}/final_validation.jsonl",
    "test": f"{base}/final_test.jsonl",
}

for split, path in files.items():
    print(f"\n{split.upper()}")
    print("Path:", path)
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

        df = pd.read_json(path, lines=True)
        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print(df.head(1).to_dict("records")[0])


TRAIN
Path: /content/SentimentAnalysis/data/processed/canonical/final_train.jsonl
Exists: False

VALIDATION
Path: /content/SentimentAnalysis/data/processed/canonical/final_validation.jsonl
Exists: False

TEST
Path: /content/SentimentAnalysis/data/processed/canonical/final_test.jsonl
Exists: False


In [14]:
%cd /content/SentimentAnalysis

print("===== DATA DIRECTORY =====")
!find data -maxdepth 4 -type f | sort

print("\n===== CANONICAL DIRECTORY =====")
!ls -lah data/processed/ 2>/dev/null || true
!ls -lah data/processed/canonical/ 2>/dev/null || true

/content/SentimentAnalysis
===== DATA DIRECTORY =====
data/validation/findings_report.json
data/validation/integrity_report.json

===== CANONICAL DIRECTORY =====


In [15]:
import os

paths = [
    "data/processed/stocktwits_train.csv",
    "data/processed/stocktwits_validation.csv",
    "data/processed/stocktwits_test.csv",
]

for p in paths:
    print(p, "->", os.path.exists(p))

data/processed/stocktwits_train.csv -> False
data/processed/stocktwits_validation.csv -> False
data/processed/stocktwits_test.csv -> False


In [16]:
# CELL 6G — Check the data-preparation code

%cd /content/SentimentAnalysis

!find src/data -maxdepth 2 -type f | sort

/content/SentimentAnalysis
src/data/build_final_dataset.py
src/data/inspect_datasets.py
src/data/preprocessing.py
src/data/stocktwits_adapter.py
src/data/validate_data.py


In [17]:
# CELL 6H — Download the StockTwits dataset

%cd /content/SentimentAnalysis

from datasets import load_dataset

print("Downloading ElKulako/stocktwits-emoji...")

ds = load_dataset("ElKulako/stocktwits-emoji")

print("\nDataset downloaded successfully!")
print(ds)


/content/SentimentAnalysis


README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

train-emoji-bear-unmodified.txt:   0%|          | 0.00/561k [00:00<?, ?B/s]

train-emoji-bull-unmodified.txt: reconstructing file:   0%|          |  0.00B / 4.56MB            

train-emoji-bull-unmodified.txt: downloading bytes:           |  0.00B            

...

test_emoji_bear.txt:   0%|          | 0.00/240k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211758 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/20761 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11984 [00:00<?, ? examples/s]


Dataset downloaded successfully!
DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 211758
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 20761
    })
    test: Dataset({
        features: ['text'],
        num_rows: 11984
    })
})


In [18]:
# CELL 6I — Inspect downloaded dataset

print("===== DATASET SPLITS =====")

for split in ds:
    print(f"{split}: {len(ds[split])} rows")

print("\n===== COLUMNS =====")
print(ds["train"].column_names)

print("\n===== FIRST TRAIN SAMPLE =====")
print(ds["train"][0])

===== DATASET SPLITS =====
train: 211758 rows
validation: 20761 rows
test: 11984 rows

===== COLUMNS =====
['text']

===== FIRST TRAIN SAMPLE =====
{'text': '  bye bye baby 😂😂😂'}


In [19]:
# CELL 6J — Save raw StockTwits dataset locally

%cd /content/SentimentAnalysis

import os

os.makedirs("data/raw/stocktwits", exist_ok=True)

for split in ["train", "validation", "test"]:
    output_path = f"data/raw/stocktwits/{split}.txt"

    with open(output_path, "w", encoding="utf-8") as f:
        for row in ds[split]:
            f.write(row["text"].replace("\n", " ") + "\n")

    print(f"{split}: saved -> {output_path}")

/content/SentimentAnalysis
train: saved -> data/raw/stocktwits/train.txt
validation: saved -> data/raw/stocktwits/validation.txt
test: saved -> data/raw/stocktwits/test.txt


In [20]:
# CELL 6K — Verify raw files

%cd /content/SentimentAnalysis

!ls -lh data/raw/stocktwits/

/content/SentimentAnalysis
total 19M
-rw-r--r-- 1 root root 1.1M Sep  7 19:10 test.txt
-rw-r--r-- 1 root root  16M Sep  7 19:10 train.txt
-rw-r--r-- 1 root root 1.7M Sep  7 19:10 validation.txt


In [21]:
# CELL 6L — Recover StockTwits labels

%cd /content/SentimentAnalysis

!python -m src.data.stocktwits_adapter

/content/SentimentAnalysis
Creating CSV from Arrow format: 100% 211/211 [00:00<00:00, 237.17ba/s]
Saved train: 210699 rows -> data/processed/stocktwits_train.csv
Creating CSV from Arrow format: 100% 21/21 [00:00<00:00, 229.98ba/s]
Saved validation: 20676 rows -> data/processed/stocktwits_validation.csv
Creating CSV from Arrow format: 100% 12/12 [00:00<00:00, 229.77ba/s]
Saved test: 11966 rows -> data/processed/stocktwits_test.csv
Report saved -> data/inspection/stocktwits_labels.json

STOCKTWITS LABEL RECOVERY COMPLETE

TRAIN:
  HF rows:       211758
  Matched:       210699
  Dropped:       1059 (conflicting labels)
  Class dist:    {'Bearish': 35843, 'Neutral': 63593, 'Bullish': 111263}
  Emoji records: 210699

VALIDATION:
  HF rows:       20761
  Matched:       20676
  Dropped:       85 (conflicting labels)
  Class dist:    {'Bearish': 4073, 'Neutral': 7496, 'Bullish': 9107}
  Emoji records: 20676

TEST:
  HF rows:       11984
  Matched:       11966
  Dropped:       18 (conflicting l

In [22]:
# CELL 6M — Build the final canonical dataset

%cd /content/SentimentAnalysis

!python -m src.data.build_final_dataset

/content/SentimentAnalysis
Saved raw snapshots: stocktwits_{train,validation,test}_raw.csv

FINAL DATASET CONSTRUCTION REPORT
Training deduplication:
  HF train rows:              211758
  After conflict-label drop:   210699 (adapter output)
  Duplicate rows removed:      119551
  Train<->test overlap removed:27
  Final training rows:         91121
  Validation (unchanged):      20676
  Test (unchanged):            11966

Class distribution (train original -> final):
  Bearish: 35843 -> 7290
  Neutral: 63593 -> 26237
  Bullish: 111263 -> 57594

Class weights (train-only, final):
  Bearish: 4.1665
  Neutral: 1.1577
  Bullish: 0.5274

Manifest written -> data/processed/experiment_manifest.json


In [23]:
# CELL 6N — Create canonical JSONL files

%cd /content/SentimentAnalysis

!python -m src.data.preprocessing

/content/SentimentAnalysis
Saved 91121 canonical rows -> data/processed/canonical/final_train.jsonl
Saved 20676 canonical rows -> data/processed/canonical/final_validation.jsonl
Saved 11966 canonical rows -> data/processed/canonical/final_test.jsonl


In [24]:
# CELL 6O — Verify canonical dataset

import os
import pandas as pd

base = "/content/SentimentAnalysis/data/processed/canonical"

files = {
    "train": f"{base}/final_train.jsonl",
    "validation": f"{base}/final_validation.jsonl",
    "test": f"{base}/final_test.jsonl",
}

for split, path in files.items():
    print(f"\n===== {split.upper()} =====")
    print("Exists:", os.path.exists(path))

    if os.path.exists(path):
        print("Size:", round(os.path.getsize(path)/(1024*1024), 2), "MB")

        df = pd.read_json(path, lines=True)

        print("Rows:", len(df))
        print("Columns:", list(df.columns))
        print("Required columns present:",
              all(c in df.columns for c in
                  ["text_without_emoji", "emoji_list", "label"]))


===== TRAIN =====
Exists: True
Size: 27.94 MB
Rows: 91121
Columns: ['id', 'split', 'original_text', 'text_without_emoji', 'emoji_list', 'num_emojis', 'label', 'label_name', 'token_count']
Required columns present: True

===== VALIDATION =====
Exists: True
Size: 6.64 MB
Rows: 20676
Columns: ['id', 'split', 'original_text', 'text_without_emoji', 'emoji_list', 'num_emojis', 'label', 'label_name', 'token_count']
Required columns present: True

===== TEST =====
Exists: True
Size: 3.93 MB
Rows: 11966
Columns: ['id', 'split', 'original_text', 'text_without_emoji', 'emoji_list', 'num_emojis', 'label', 'label_name', 'token_count']
Required columns present: True


In [25]:
# E2-1A — Fix torchvision compatibility

%cd /content/SentimentAnalysis

!pip uninstall -y torchvision
!pip install torchvision==0.24.0

/content/SentimentAnalysis
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 6.0 MB/s eta 0:00:00
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
  Attempting uninstall: nvidia-nvshmem-cu12
    Found existing installation: nvidia-nvshmem-cu12 3.4.5
    Uninstalling nvidia-nvshmem-cu12-3.4.5:
      Successfully uninstalled nvidia-nvshmem-cu12-3

In [1]:
# E2-1B — Verify PyTorch / torchvision

import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.9.0+cu128
Torchvision: 0.24.0+cu128
CUDA available: True


In [2]:
# E2-1D — Remove incompatible torchaudio

!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


In [3]:
# E2-1E — Verify PyTorch environment

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch: 2.9.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# E2-1F — Generate pretrained emoji embeddings

%cd /content/SentimentAnalysis

import os
os.environ["PYTHONPATH"] = "/content/SentimentAnalysis"

!python -m src.train_e2_pretrain

/content/SentimentAnalysis
Starting TweetEval emoji pretraining...
Using device: cuda
README.md: 100% 23.9k/23.9k [00:00<00:00, 27.1MB/s]

emoji/train-00000-of-00001.parquet: downloading bytes:  28% 728k/2.61M [00:00<00:02, 852kB/s]
emoji/train-00000-of-00001.parquet: downloading bytes: 100% 2.60M/2.60M [00:00<00:00, 3.00MB/s,  257kB/s  ]
emoji/train-00000-of-00001.parquet: reconstructing file: 100% 2.61M/2.61M [00:00<00:00, 3.01MB/s,  258kB/s  ]

emoji/test-00000-of-00001.parquet: downloading bytes:   4% 131k/3.05M [00:00<00:18, 162kB/s]
emoji/test-00000-of-00001.parquet: downloading bytes: 100% 3.02M/3.02M [00:00<00:00, 3.18MB/s,  295kB/s  ]
emoji/test-00000-of-00001.parquet: reconstructing file: 100% 3.05M/3.05M [00:00<00:00, 3.20MB/s,  298kB/s  ]

emoji/validation-00000-of-00001.parquet: downloading bytes:   0% 0.00/282k [00:00<?, ?B/s]
emoji/validation-00000-of-00001.parquet: downloading bytes: 100% 279k/279k [00:00<00:00, 493kB/s, 27.7kB/s  ]
emoji/validation-00000-of-00001.parqu

In [10]:
%cd /content/SentimentAnalysis

import torch
import os

path = "models/emoji_embeddings/pretrained_emoji_32d.pt"

assert os.path.exists(path), f"Missing: {path}"

state = torch.load(path, map_location="cpu")

print("Keys:", state.keys())
print("Embedding shape:", state["emoji_embedding"].shape)
print("Embedding dimension:", state.get("emoji_dim"))
print("Pretrained vocab size:", state.get("vocab_size"))

assert state["emoji_embedding"].shape == (20, 32), \
    "ERROR: pretrained embedding must be [20, 32]"

print("\n✅ Pretrained TweetEval embedding is valid.")

/content/SentimentAnalysis
Keys: dict_keys(['emoji_embedding', 'emoji_dim', 'vocab_size', 'label_names'])
Embedding shape: torch.Size([20, 32])
Embedding dimension: 32
Pretrained vocab size: 20

✅ Pretrained TweetEval embedding is valid.


In [11]:
%cd /content/SentimentAnalysis

import json
import os

vocab_path = "models/emoji_embeddings/emoji_vocab.json"

assert os.path.exists(vocab_path), f"Missing: {vocab_path}"

with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)

print("Vocabulary keys:", vocab.keys())

emoji_to_id = vocab["emoji_to_id"]

print("StockTwits vocabulary size:", len(emoji_to_id))
print("First few entries:", list(emoji_to_id.items())[:20])

assert len(emoji_to_id) == 1610, \
    f"Expected 1610 emojis, found {len(emoji_to_id)}"

print("\n✅ StockTwits vocabulary verified.")

/content/SentimentAnalysis
Vocabulary keys: dict_keys(['emoji_to_id', 'id_to_emoji', 'vocab_size', 'unk_id', 'counts'])
StockTwits vocabulary size: 1610
First few entries: [('<UNK>', 0), ('😂', 1), ('🚀', 2), ('🤣', 3), ('🔥', 4), ('💎', 5), ('🤑', 6), ('🤡', 7), ('🤔', 8), ('💰', 9), ('🐻', 10), ('👀', 11), ('😎', 12), ('📈', 13), ('💪', 14), ('😆', 15), ('👍', 16), ('😅', 17), ('😉', 18), ('✅', 19)]

✅ StockTwits vocabulary verified.


In [12]:
%cd /content/SentimentAnalysis

import json
import torch

# Load StockTwits vocabulary
with open("models/emoji_embeddings/emoji_vocab.json", "r", encoding="utf-8") as f:
    vocab = json.load(f)

stocktwits_emoji_to_id = vocab["emoji_to_id"]

# Load TweetEval pretrained state
state = torch.load(
    "models/emoji_embeddings/pretrained_emoji_32d.pt",
    map_location="cpu"
)

tweet_eval_emoji_to_id = state.get("emoji_to_id")

print("TweetEval mapping found:", tweet_eval_emoji_to_id is not None)

if tweet_eval_emoji_to_id is not None:
    print("TweetEval vocabulary:", tweet_eval_emoji_to_id)

    overlap = sorted(
        set(tweet_eval_emoji_to_id.keys()) &
        set(stocktwits_emoji_to_id.keys())
    )

    print("\nTweetEval emojis:", len(tweet_eval_emoji_to_id))
    print("StockTwits emojis:", len(stocktwits_emoji_to_id))
    print("Exact overlap:", len(overlap))

    print("\nOverlapping emojis:")
    print(overlap)
else:
    print(
        "\n⚠️ pretrained artifact does not contain emoji_to_id."
        "\nWe need to inspect how train_e2_pretrain.py defines the ordering."
    )

/content/SentimentAnalysis
TweetEval mapping found: False

⚠️ pretrained artifact does not contain emoji_to_id.
We need to inspect how train_e2_pretrain.py defines the ordering.


In [13]:
%cd /content/SentimentAnalysis

import torch

state = torch.load(
    "models/emoji_embeddings/pretrained_emoji_32d.pt",
    map_location="cpu"
)

print("Saved keys:")
for k, v in state.items():
    if torch.is_tensor(v):
        print(k, "->", v.shape)
    else:
        print(k, "->", v)

print("\nExpected TweetEval emoji order:")
tweet_eval_emojis = [
    '❤', '😍', '😂', '💕', '🔥',
    '😊', '😎', '✨', '💙', '😘',
    '📷', '🇺🇸', '☀', '💜', '😉',
    '💯', '😁', '🎄', '📸', '😜'
]

for i, emoji in enumerate(tweet_eval_emojis):
    print(i, emoji)

/content/SentimentAnalysis
Saved keys:
emoji_embedding -> torch.Size([20, 32])
emoji_dim -> 32
vocab_size -> 20
label_names -> ['❤', '😍', '😂', '💕', '🔥', '😊', '😎', '✨', '💙', '😘', '📷', '🇺🇸', '☀', '💜', '😉', '💯', '😁', '🎄', '📸', '😜']

Expected TweetEval emoji order:
0 ❤
1 😍
2 😂
3 💕
4 🔥
5 😊
6 😎
7 ✨
8 💙
9 😘
10 📷
11 🇺🇸
12 ☀
13 💜
14 😉
15 💯
16 😁
17 🎄
18 📸
19 😜


In [14]:
%cd /content/SentimentAnalysis

print("===== train_e2.py =====")
!sed -n '1,280p' src/train_e2.py

print("\n===== e2_concat_fusion.py =====")
!sed -n '1,320p' src/models/e2_concat_fusion.py

/content/SentimentAnalysis
===== train_e2.py =====
"""E2 — text + pretrained emoji embedding, concatenation fusion.

MIRRORS E0/E1 exactly (seed, class weights, optimizer, batch, max_length, metrics,
model-selection criterion) except it uses PRETRAINED emoji embeddings (frozen)
learned from the TweetEval emoji prediction task, fused by concatenation.

Controlled-experiment contract:
- Same train/validation/test examples and labels as E0/E1.
- Text branch receives ONLY ``text_without_emoji``.
- Emoji branch receives ONLY ``emoji_list`` (pretrained frozen embedding).
- ``original_text`` is never passed to BERT.
- Emoji embeddings are pretrained on TweetEval emoji prediction task (20 classes)
  and frozen during E2 training.

This script is PREPARED but must NOT be trained until approved.
"""

from __future__ import annotations

import hashlib
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
imp

In [15]:
# CELL 12 — E2 implementation validation

%cd /content/SentimentAnalysis

import torch
import json
import os
import inspect

print("=" * 70)
print("E2 IMPLEMENTATION VALIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Check pretrained embedding
# ------------------------------------------------------------
pretrained_path = "models/emoji_embeddings/pretrained_emoji_32d.pt"

state = torch.load(pretrained_path, map_location="cpu")

print("\n[1] PRETRAINED EMBEDDING")
print("Shape:", tuple(state["emoji_embedding"].shape))
print("Dimension:", state.get("emoji_dim"))
print("Vocab size:", state.get("vocab_size"))
print("Keys:", list(state.keys()))

# ------------------------------------------------------------
# 2. Check StockTwits vocabulary
# ------------------------------------------------------------
with open(
    "models/emoji_embeddings/emoji_vocab.json",
    "r",
    encoding="utf-8"
) as f:
    vocab = json.load(f)

emoji_to_id = vocab["emoji_to_id"]

print("\n[2] STOCKTWITS VOCABULARY")
print("Vocabulary size:", len(emoji_to_id))
print("UNK ID:", emoji_to_id.get("<UNK>"))

# ------------------------------------------------------------
# 3. Check source code for dangerous hardcoding
# ------------------------------------------------------------
print("\n[3] CHECKING train_e2.py")

with open("src/train_e2.py", "r", encoding="utf-8") as f:
    train_code = f.read()

for pattern in [
    "vocab_size=32",
    "nn.Embedding(32, 32)",
    "emoji_vocab_size=32",
    ".vocab_size"
]:
    print(
        f"{pattern}:",
        "FOUND ⚠️" if pattern in train_code else "not found ✅"
    )

print("\n[4] CHECKING e2_concat_fusion.py")

with open(
    "src/models/e2_concat_fusion.py",
    "r",
    encoding="utf-8"
) as f:
    model_code = f.read()

for pattern in [
    "nn.Embedding(32, 32)",
    "nn.Embedding(emoji_dim, vocab_size)",
    "emoji_encoder(e_ids, e_mask)",
    "emoji_encoder.vocab_size"
]:
    print(
        f"{pattern}:",
        "FOUND ⚠️" if pattern in model_code else "not found ✅"
    )

# ------------------------------------------------------------
# 5. Check train_e2 imports
# ------------------------------------------------------------
print("\n[5] IMPORT TEST")

try:
    import src.train_e2
    print("src.train_e2 import: ✅")
except Exception as e:
    print("src.train_e2 import: ❌")
    print(type(e).__name__, ":", e)

try:
    import src.models.e2_concat_fusion
    print("e2_concat_fusion import: ✅")
except Exception as e:
    print("e2_concat_fusion import: ❌")
    print(type(e).__name__, ":", e)

# ------------------------------------------------------------
# 6. GPU
# ------------------------------------------------------------
print("\n[6] GPU")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

/content/SentimentAnalysis
E2 IMPLEMENTATION VALIDATION

[1] PRETRAINED EMBEDDING
Shape: (20, 32)
Dimension: 32
Vocab size: 20
Keys: ['emoji_embedding', 'emoji_dim', 'vocab_size', 'label_names']

[2] STOCKTWITS VOCABULARY
Vocabulary size: 1610
UNK ID: 0

[3] CHECKING train_e2.py
vocab_size=32: FOUND ⚠️
nn.Embedding(32, 32): not found ✅
emoji_vocab_size=32: FOUND ⚠️
.vocab_size: not found ✅

[4] CHECKING e2_concat_fusion.py
nn.Embedding(32, 32): not found ✅
nn.Embedding(emoji_dim, vocab_size): not found ✅
emoji_encoder(e_ids, e_mask): not found ✅
emoji_encoder.vocab_size: not found ✅

[5] IMPORT TEST
src.train_e2 import: ✅
e2_concat_fusion import: ✅

[6] GPU
PyTorch: 2.9.0+cu128
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB

VALIDATION COMPLETE


In [16]:
# CELL 13 — Verify final E2 embedding construction

%cd /content/SentimentAnalysis

import torch
import json
import os

# -----------------------------
# Load StockTwits vocabulary
# -----------------------------
with open(
    "models/emoji_embeddings/emoji_vocab.json",
    "r",
    encoding="utf-8"
) as f:
    vocab = json.load(f)

emoji_to_id = vocab["emoji_to_id"]

print("StockTwits vocab:", len(emoji_to_id))

assert len(emoji_to_id) == 1610

# -----------------------------
# Load TweetEval embedding
# -----------------------------
state = torch.load(
    "models/emoji_embeddings/pretrained_emoji_32d.pt",
    map_location="cpu"
)

pretrained = state["emoji_embedding"]

print("TweetEval embedding:", tuple(pretrained.shape))

assert pretrained.shape == (20, 32)

# -----------------------------
# TweetEval ordering
# -----------------------------
tweet_eval_emojis = [
    '❤', '😍', '😂', '💕', '🔥',
    '😊', '😎', '✨', '💙', '😘',
    '📷', '🇺🇸', '☀', '💜', '😉',
    '💯', '😁', '🎄', '📸', '😜'
]

# -----------------------------
# Check overlap
# -----------------------------
overlap = [
    e for e in tweet_eval_emojis
    if e in emoji_to_id
]

print("TweetEval emojis:", len(tweet_eval_emojis))
print("Exact overlap:", len(overlap))
print("Overlap emojis:", overlap)

assert len(overlap) == 19

print("\n✅ Vocabulary relationship is correct.")

/content/SentimentAnalysis
StockTwits vocab: 1610
TweetEval embedding: (20, 32)
TweetEval emojis: 20
Exact overlap: 19
Overlap emojis: ['❤', '😍', '😂', '💕', '🔥', '😊', '😎', '✨', '💙', '😘', '📷', '🇺🇸', '💜', '😉', '💯', '😁', '🎄', '📸', '😜']

✅ Vocabulary relationship is correct.


In [17]:
# CELL 14 — Build final StockTwits E2 emoji embedding

import torch
import json
import os
import random
import numpy as np

SEED = 42
EMBED_DIM = 32

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# -----------------------------
# Load vocab
# -----------------------------
with open(
    "models/emoji_embeddings/emoji_vocab.json",
    "r",
    encoding="utf-8"
) as f:
    vocab = json.load(f)

emoji_to_id = vocab["emoji_to_id"]

vocab_size = len(emoji_to_id)

# -----------------------------
# Load pretrained TweetEval
# -----------------------------
state = torch.load(
    "models/emoji_embeddings/pretrained_emoji_32d.pt",
    map_location="cpu"
)

pretrained = state["emoji_embedding"]

# -----------------------------
# TweetEval order
# -----------------------------
tweet_eval_emojis = [
    '❤', '😍', '😂', '💕', '🔥',
    '😊', '😎', '✨', '💙', '😘',
    '📷', '🇺🇸', '☀', '💜', '😉',
    '💯', '😁', '🎄', '📸', '😜'
]

# -----------------------------
# Random initialization
# -----------------------------
generator = torch.Generator()
generator.manual_seed(SEED)

final_embedding = torch.randn(
    vocab_size,
    EMBED_DIM,
    generator=generator
)

# -----------------------------
# Transfer exact overlaps
# -----------------------------
transferred = []

for tweet_id, emoji in enumerate(tweet_eval_emojis):

    if emoji in emoji_to_id:

        stock_id = emoji_to_id[emoji]

        final_embedding[stock_id] = pretrained[tweet_id]

        transferred.append(emoji)

print("Final embedding shape:", tuple(final_embedding.shape))
print("Pretrained transferred:", len(transferred))
print("Randomly initialized:", vocab_size - len(transferred))

print("\nTransferred emojis:")
print(transferred)

assert final_embedding.shape == (1610, 32)
assert len(transferred) == 19

print("\n✅ Final E2 embedding = 1610 × 32")
print("✅ 19 pretrained rows transferred")
print("✅ 1591 rows randomly initialized")

Final embedding shape: (1610, 32)
Pretrained transferred: 19
Randomly initialized: 1591

Transferred emojis:
['❤', '😍', '😂', '💕', '🔥', '😊', '😎', '✨', '💙', '😘', '📷', '🇺🇸', '💜', '😉', '💯', '😁', '🎄', '📸', '😜']

✅ Final E2 embedding = 1610 × 32
✅ 19 pretrained rows transferred
✅ 1591 rows randomly initialized


In [18]:
# CELL 15 — Save final E2 emoji embedding

import os
import torch

os.makedirs("models/emoji_embeddings", exist_ok=True)

output_path = (
    "models/emoji_embeddings/"
    "stocktwits_emoji_embedding_e2_1610x32.pt"
)

torch.save(
    {
        "emoji_embedding": final_embedding,
        "emoji_to_id": emoji_to_id,
        "embedding_dim": 32,
        "vocab_size": 1610,
        "seed": 42,
        "pretrained_source": "TweetEval",
        "tweet_eval_pretrained_count": 19,
        "random_initialized_count": 1591,
        "tweet_eval_vocab_size": 20,
    },
    output_path
)

print("Saved:", output_path)
print("Size:", os.path.getsize(output_path) / (1024 * 1024), "MB")

Saved: models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt
Size: 0.22722911834716797 MB


In [19]:
# CELL 16 — Verify saved E2 embedding

import torch

path = (
    "models/emoji_embeddings/"
    "stocktwits_emoji_embedding_e2_1610x32.pt"
)

saved = torch.load(path, map_location="cpu")

print("Keys:", saved.keys())
print("Shape:", tuple(saved["emoji_embedding"].shape))
print("Vocab size:", saved["vocab_size"])
print("Embedding dim:", saved["embedding_dim"])
print("Pretrained count:", saved["tweet_eval_pretrained_count"])
print("Random count:", saved["random_initialized_count"])
print("Seed:", saved["seed"])

assert saved["emoji_embedding"].shape == (1610, 32)
assert saved["tweet_eval_pretrained_count"] == 19
assert saved["random_initialized_count"] == 1591

print("\n✅ E2 embedding artifact verified.")

Keys: dict_keys(['emoji_embedding', 'emoji_to_id', 'embedding_dim', 'vocab_size', 'seed', 'pretrained_source', 'tweet_eval_pretrained_count', 'random_initialized_count', 'tweet_eval_vocab_size'])
Shape: (1610, 32)
Vocab size: 1610
Embedding dim: 32
Pretrained count: 19
Random count: 1591
Seed: 42

✅ E2 embedding artifact verified.


In [20]:
# CELL 17 — E2 model construction + GPU smoke test

%cd /content/SentimentAnalysis

import torch
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

# Load a tiny sample
train_path = "data/processed/canonical/final_train.jsonl"

df = pd.read_json(train_path, lines=True)

sample = df.head(2)

print("\nSample:")
print(
    sample[
        ["text_without_emoji", "emoji_list", "label"]
    ].to_dict("records")
)

print("\n✅ Dataset can be loaded.")
print("Ready for E2 model smoke test.")

/content/SentimentAnalysis
Device: cuda
GPU: Tesla T4
VRAM: 14.56 GB

Sample:
[{'text_without_emoji': '  bye bye baby ', 'emoji_list': ['😂'], 'label': 0}, {'text_without_emoji': ' all you patient ones imagine if you had exercised patience before buying ', 'emoji_list': ['😂'], 'label': 0}]

✅ Dataset can be loaded.
Ready for E2 model smoke test.


In [21]:
# CELL 18 — Inspect the current E2 training script

%cd /content/SentimentAnalysis

!grep -n -E "vocab_size|emoji_vocab|Embedding|pretrained|batch_size|DataLoader|autocast|GradScaler" src/train_e2.py

/content/SentimentAnalysis
1:"""E2 — text + pretrained emoji embedding, concatenation fusion.
10:- Emoji branch receives ONLY ``emoji_list`` (pretrained frozen embedding).
12:- Emoji embeddings are pretrained on TweetEval emoji prediction task (20 classes)
39:from torch.utils.data import DataLoader, Dataset, TensorDataset
45:    build_emoji_vocab,
63:def featurize_texts(texts, tokenizer, encoder, max_length, batch_size, device):
67:        for i in tqdm(range(0, len(texts), batch_size), desc="Text featurize"):
68:            batch = texts[i:i + batch_size]
111:    batch_size = train_cfg["batch_size"]
116:    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
117:    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)
180:                "emoji_vocab_size": 32,
221:    vocab = build_emoji_vocab(train_df["emoji_list"].tolist())
222:    print(f"Emoji vocab size (train-only): {vocab['vocab_size']}")
223:    save_vocab_pa

In [22]:
# CELL 19 — Show complete train_e2.py

%cd /content/SentimentAnalysis

!cat src/train_e2.py

/content/SentimentAnalysis
"""E2 — text + pretrained emoji embedding, concatenation fusion.

MIRRORS E0/E1 exactly (seed, class weights, optimizer, batch, max_length, metrics,
model-selection criterion) except it uses PRETRAINED emoji embeddings (frozen)
learned from the TweetEval emoji prediction task, fused by concatenation.

Controlled-experiment contract:
- Same train/validation/test examples and labels as E0/E1.
- Text branch receives ONLY ``text_without_emoji``.
- Emoji branch receives ONLY ``emoji_list`` (pretrained frozen embedding).
- ``original_text`` is never passed to BERT.
- Emoji embeddings are pretrained on TweetEval emoji prediction task (20 classes)
  and frozen during E2 training.

This script is PREPARED but must NOT be trained until approved.
"""

from __future__ import annotations

import hashlib
import json
import os
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from 

In [23]:
pretrained_path = (
    "models/emoji_embeddings/"
    "stocktwits_emoji_embedding_e2_1610x32.pt"
)

In [24]:
embedding_matrix = saved["emoji_embedding"]
vocab_size = embedding_matrix.shape[0]  # 1610
emoji_dim = embedding_matrix.shape[1]   # 32

In [29]:
%cd /content/SentimentAnalysis
!git status
!git pull origin main

/content/SentimentAnalysis
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/emoji_embeddings/tweeteval_emoji_vocab.json

nothing added to commit but untracked files present (use "git add" to track)
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 4), reused 6 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.02 KiB | 1.01 MiB/s, done.
From https://github.com/Chetnapadhi/SentimentAnalysis
 * branch            main       -> FETCH_HEAD
   56ff002..cec8561  main       -> origin/main
Updating 56ff002..cec8561
Fast-forward
 src/models/e2_concat_fusion.py | 128 +++++++++++++++++++++++++++++++++--------
 src/train_e2.py                | 117 ++++++++++++++++++++++++++++++-------
 2 files changed, 199 insertions(+), 46 deletions(-)


In [30]:
%cd /content/SentimentAnalysis
!git status

/content/SentimentAnalysis
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	models/emoji_embeddings/tweeteval_emoji_vocab.json

nothing added to commit but untracked files present (use "git add" to track)


In [31]:
!git add models/emoji_embeddings/tweeteval_emoji_vocab.json
!git commit -m "Add TweetEval emoji vocabulary"
!git push origin main

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@9e92edf20d5e.(none)')
fatal: could not read Username for 'https://github.com': No such device or address


In [32]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   models/emoji_embeddings/tweeteval_emoji_vocab.json



In [33]:
%cd /content/SentimentAnalysis

!grep -nE "vocab_size|emoji_vocab_size|stocktwits_emoji_embedding|pretrained|Embedding" src/train_e2.py

/content/SentimentAnalysis
1:"""E2 — text + pretrained emoji embedding, concatenation fusion.
10:- Emoji branch receives ONLY ``emoji_list`` (pretrained frozen embedding).
12:- Emoji embeddings are pretrained on TweetEval emoji prediction task (20 classes)
200:                "emoji_vocab_size": vocab['vocab_size'],  # Use actual vocab size
242:    print(f"Emoji vocab size (train-only): {vocab['vocab_size']}")
259:    # Load the final transferred embedding artifact (1610x32, 19 pretrained + 1591 random)
260:    pretrained_emoji_path = "models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt"
261:    if not os.path.exists(pretrained_emoji_path):
263:            f"Pretrained emoji embedding not found at {pretrained_emoji_path}. "
267:    pretrained_emoji = torch.load(pretrained_emoji_path, map_location="cpu")
268:    print(f"Loaded pretrained emoji embedding: {pretrained_emoji_path}")
269:    print(f"  Embedding shape: {pretrained_emoji['emoji_embedding'].shape}")
270:    print(

In [34]:
!grep -nE "vocab_size|Embedding|emoji_embedding|forward" src/models/e2_concat_fusion.py

57:    def __init__(self, vocab_size: int, emoji_dim: int = 32, num_labels: int = 20):
61:        # Emoji embedding (to be pretrained) - FIXED: vocab_size, emoji_dim order
62:        self.emoji_encoder = nn.Embedding(vocab_size, emoji_dim, padding_idx=None)
78:    def forward(self, input_ids, attention_mask):
93:    def get_emoji_embeddings(self):
135:def pretrain_emoji_embeddings():
151:    print(f"TweetEval emoji vocab size: {vocab['vocab_size']}")
154:    os.makedirs("models/emoji_embeddings", exist_ok=True)
156:    save_vocab(vocab, "models/emoji_embeddings/tweeteval_emoji_vocab.json")
164:    # Model - vocab_size is number of TweetEval emoji classes (20), not embedding dim
166:        vocab_size=num_tweeteval_emojis,  # 20 TweetEval emoji classes
216:                "emoji_embedding": model.emoji_encoder.weight.data.clone(),  # [20, 32]
218:                "vocab_size": num_tweeteval_emojis,  # 20
222:            torch.save(best_state, "models/emoji_embeddings/pretrained_emoji_20x

In [35]:
import torch
from pathlib import Path

p = Path("models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt")

print("Exists:", p.exists())

if p.exists():
    obj = torch.load(p, map_location="cpu")
    print("Embedding shape:", obj["emoji_embedding"].shape)
    print("Vocab size:", obj["vocab_size"])
    print("Embedding dim:", obj["embedding_dim"])
    print("Pretrained:", obj["tweet_eval_pretrained_count"])
    print("Random:", obj["random_initialized_count"])

Exists: True
Embedding shape: torch.Size([1610, 32])
Vocab size: 1610
Embedding dim: 32
Pretrained: 19
Random: 1591


In [36]:
import src.models.e2_concat_fusion
import src.train_e2

print("E2 imports: OK")

E2 imports: OK


In [37]:
%cd /content/SentimentAnalysis

/content/SentimentAnalysis


In [38]:
!python -m src.train_e2

E2 is prepared but disabled. Set RUN_E2=1 to train (after approval).


In [40]:
%cd /content/SentimentAnalysis

from pathlib import Path
import shutil

src = Path("src/train_e2.py")
backup = Path("src/train_e2_backup_before_guard_fix.py")

shutil.copy(src, backup)

print("Backup created:", backup)

/content/SentimentAnalysis
Backup created: src/train_e2_backup_before_guard_fix.py


In [41]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/train_e2.py").read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if i >= 300:
        print(f"{i:3}: {line}")

/content/SentimentAnalysis
300:     # Create text encoder (frozen BERT)
301:     tokenizer = AutoTokenizer.from_pretrained(model_name)
302:     text_model = E2ConcatFusionModel(
303:         model_name=model_name,
304:         text_dim=768,
305:         emoji_dim=pretrained_emoji['embedding_dim'],  # 32
306:         num_labels=n_classes,
307:         dropout=cfg["classifier_head"]["dropout"],
308:         classifier_hidden=cfg["classifier_head"]["classifier_hidden"],
309:         freeze_encoder=True,
310:         emoji_vocab_size=pretrained_emoji['vocab_size'],  # 1610
311:         max_emojis=max_emojis,
312:         pretrained_emoji_path="",  # We'll load weights manually after model creation
313:     )
314:     
315:     # Load pretrained emoji embeddings into the model
316:     pretrained_emoji_data = torch.load("models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt", map_location="cpu")
317:     pretrained_embedding = pretrained_emoji['emoji_embedding']  # [1610, 32]
318

In [42]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("src/train_e2.py")
text = path.read_text(encoding="utf-8")

old = '''    # Train (NOT EXECUTED here — guarded)
    print("\\n[E2 PREPARED] Training is NOT executed by default. To train: RUN_E2=1 python -m src.train_e2")
    assert False, "E2 training is disabled until approved. Set RUN_E2=1 to enable."
'''

new = '''    # Train E2
    print("\\n[E2] Starting training...")

    out_dir = "results/E2"
    os.makedirs(out_dir, exist_ok=True)

    metrics = train_e2(
        model=model,
        train_ds=train_ds,
        val_ds=val_ds,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
        out_dir=out_dir,
    )

    print("\\n[E2] Training and validation complete.")
    print(metrics)
'''

if old not in text:
    print("❌ Expected guard block not found.")
    print("No changes were made.")
else:
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("✅ E2 training guard removed successfully.")

/content/SentimentAnalysis
❌ Expected guard block not found.
No changes were made.


In [43]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/train_e2.py").read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if i >= 340:
        print(f"{i:3}: {line}")

/content/SentimentAnalysis
340:         if os.path.exists(path):
341:             return np.load(path)
342:         emb = featurize_texts(split_df["text_without_emoji"].tolist(), tokenizer,
343:                               encoder, max_length, train_cfg["batch_size"], device)
344:         np.save(path, emb)
345:         return emb
346: 
347:     X_train = get_emb(train_df, "train")
348:     X_val = get_emb(val_df, "validation")
349:     X_test = get_emb(test_df, "test")
350: 
351:     # Encode emojis
352:     def encode_split(df):
353:         ids, masks = [], []
354:         for emojis in df["emoji_list"].tolist():
355:             e_ids, e_mask = encode_emoji_list(emojis, vocab["emoji_to_id"], max_emojis)
356:             ids.append(e_ids)
357:             masks.append(e_mask)
358:         return np.array(ids), np.array(masks)
359: 
360:     e_train_ids, e_train_mask = encode_split(train_df)
361:     e_val_ids, e_val_mask = encode_split(val_df)
362:     e_test_ids, e_test_mask = en

In [45]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("src/train_e2.py")
text = path.read_text(encoding="utf-8")

old = '''    # Train (NOT EXECUTED here — guarded)
    print("\\n[E2 PREPARED] Training is NOT executed by default.")
    print("To train: RUN_E2=1 python -m src.train_e2")
    assert False, "E2 training is disabled until approved. Set RUN_E2=1 to enable."
'''

new = '''    # Train E2
    print("\\n[E2] Starting training...")

    out_dir = "results/E2"
    os.makedirs(out_dir, exist_ok=True)

    metrics = train_e2(
        model=text_model,
        train_ds=train_ds,
        val_ds=val_ds,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
        out_dir=out_dir,
    )

    print("\\n[E2] Training and validation complete.")
    print(metrics)
'''

if old in text:
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("✅ E2 training block patched successfully.")
else:
    print("❌ Exact block still not found.")

/content/SentimentAnalysis
✅ E2 training block patched successfully.


In [46]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/train_e2.py").read_text(encoding="utf-8").splitlines()

for i, line in enumerate(lines, 1):
    if i >= 360:
        print(f"{i:3}: {line}")

/content/SentimentAnalysis
360:     e_train_ids, e_train_mask = encode_split(train_df)
361:     e_val_ids, e_val_mask = encode_split(val_df)
362:     e_test_ids, e_test_mask = encode_split(test_df)
363: 
364:     train_ds = EmojiDataset(X_train, e_train_ids, e_train_mask, train_df["label"].to_numpy())
365:     val_ds = EmojiDataset(X_val, e_val_ids, e_val_mask, val_df["label"].to_numpy())
366:     test_ds = EmojiDataset(X_test, e_test_ids, e_test_mask, test_df["label"].to_numpy())
367: 
368:     # Train E2
369:     print("\n[E2] Starting training...")
370: 
371:     out_dir = "results/E2"
372:     os.makedirs(out_dir, exist_ok=True)
373: 
374:     metrics = train_e2(
375:         model=text_model,
376:         train_ds=train_ds,
377:         val_ds=val_ds,
378:         cfg=cfg,
379:         device=device,
380:         class_weights=class_weights,
381:         out_dir=out_dir,
382:     )
383: 
384:     print("\n[E2] Training and validation complete.")
385:     print(metrics)
386: 
387: 

In [47]:
if os.environ.get("RUN_E2") == "1":
    main()

In [49]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/models/e2_concat_fusion.py").read_text(encoding="utf-8").splitlines()

for i in range(390, 425):
    print(f"{i+1:3}: {lines[i]}")

/content/SentimentAnalysis
391:     def set_stocktwits_vocab(self, stocktwits_emoji_to_id: dict):
392:         """Set the StockTwits emoji vocabulary and initialize pretrained embeddings."""
393:         if not hasattr(self, '_pretrained_emb'):
394:             return
395:         
396:         # Create mapping from TweetEval emoji to StockTwits ID
397:         tweeteval_emoji_to_id = self._tweeteval_emoji_to_id_dict
398:         pretrained_emb = self._pretrained_emb  # [20, 32]
399:         
400:         overlap_count = 0
401:         with torch.no_grad():
402:             for tweeteval_emoji, tweeteval_id in self._tweeteval_emoji_to_id_dict.items():
403:                 if tweeteval_emoji in self.emoji_to_id:
404:                     stocktwits_id = self.emoji_to_id[tweeteval_emoji]
405:                     # Copy pretrained vector
406:                     self.emoji_encoder.weight.data[stocktwits_id] = self._pretrained_emb[tweeteval_id]
407:                     overlap_count += 1
40

In [50]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("src/models/e2_concat_fusion.py")
text = path.read_text(encoding="utf-8")

start = text.index("    def emoji_forward(self, emoji_ids, emoji_masks):")

# Find the next method after emoji_forward
next_def = text.find("\n    def ", start + 10)

old_block = text[start:next_def if next_def != -1 else len(text)]

new_block = '''    def emoji_forward(self, emoji_ids, emoji_masks):
        """
        Convert emoji IDs into a fixed-size representation.

        Args:
            emoji_ids:    [B, max_emojis]
            emoji_masks:  [B, max_emojis]

        Returns:
            emoji_rep: [B, emoji_dim]
        """
        emoji_embeddings = self.emoji_encoder(emoji_ids)  # [B, max_emojis, emoji_dim]

        mask = emoji_masks.unsqueeze(-1).float()  # [B, max_emojis, 1]

        summed = (emoji_embeddings * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)

        emoji_rep = summed / counts  # [B, emoji_dim]

        return emoji_rep
'''

path.write_text(text[:start] + new_block + text[next_def if next_def != -1 else len(text):],
                encoding="utf-8")

print("✅ emoji_forward() fixed.")

/content/SentimentAnalysis
✅ emoji_forward() fixed.


In [53]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("src/train_e2.py")
text = path.read_text(encoding="utf-8")

old = '"emoji_vocab_size": vocab[\'vocab_size\'],  # Use actual vocab size'
new = '"emoji_vocab_size": len(emoji_to_id),  # Use actual StockTwits vocabulary size'

if old in text:
    text = text.replace(old, new)
    path.write_text(text, encoding="utf-8")
    print("✅ Fixed undefined vocab variable.")
else:
    print("❌ Exact line not found. Inspecting nearby code...")
    lines = text.splitlines()
    for i in range(190, 206):
        if i < len(lines):
            print(f"{i+1:3}: {lines[i]}")

/content/SentimentAnalysis
✅ Fixed undefined vocab variable.


In [54]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/train_e2.py").read_text(encoding="utf-8").splitlines()

for i in range(190, 206):
    if i < len(lines):
        print(f"{i+1:3}: {lines[i]}")

/content/SentimentAnalysis
191:         print(f"Epoch {epoch}/{epochs} | TrainLoss {train_loss:.4f} | ValLoss {val_loss:.4f} | "
192:               f"ValAcc {val_acc:.4f} | ValMacroF1 {val_mf1:.4f}")
193: 
194:         if val_mf1 > best_val_f1:
195:             best_val_f1 = val_mf1
196:             best_epoch = epoch
197:             no_improve = 0
198:             torch.save({
199:                 "classifier_state": model.classifier.state_dict(),
200:                 "emoji_vocab_size": len(emoji_to_id),  # Use actual StockTwits vocabulary size
201:                 "best_val_f1": val_mf1,
202:                 "best_epoch": epoch,
203:                 "class_weights": class_weights.tolist(),
204:             }, os.path.join(out_dir, "best_model.pt"))
205:         else:
206:             no_improve += 1


In [57]:
%cd /content/SentimentAnalysis

from pathlib import Path
import ast
import re

print("=" * 80)
print("                    E2 DEEP CODE AUDIT")
print("=" * 80)

# ------------------------------------------------------------------
# FILES
# ------------------------------------------------------------------
files_to_check = [
    "src/train_e2.py",
    "src/models/e2_concat_fusion.py",
    "config.yaml",
    "requirements.txt",
]

for f in files_to_check:
    p = Path(f)
    print(f"\n{'='*80}")
    print(f"FILE: {f}")
    print(f"{'='*80}")

    if not p.exists():
        print("❌ MISSING")
        continue

    print(f"✅ Exists: {p.stat().st_size:,} bytes")

# ------------------------------------------------------------------
# 1. SEARCH FOR STALE / SUSPICIOUS REFERENCES
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("1. STALE / SUSPICIOUS REFERENCES")
print("=" * 80)

patterns = [
    "vocab[",
    "vocab.",
    "emoji_to_id",
    "emoji_vocab",
    "vocab_size=32",
    "emoji_vocab_size=32",
    "Embedding(32",
    "pretrained_emoji_32d.pt",
    "pretrained_emoji",
    "input_ids=input_ids",
    "attention_mask=attention_mask",
    "self.emoji_forward(",
    "RUN_E2",
    "assert False",
]

for f in ["src/train_e2.py", "src/models/e2_concat_fusion.py"]:
    print(f"\n--- {f} ---")
    text = Path(f).read_text(encoding="utf-8")

    found_any = False

    for pattern in patterns:
        matches = []
        for i, line in enumerate(text.splitlines(), 1):
            if pattern in line:
                matches.append((i, line.strip()))

        if matches:
            found_any = True
            print(f"\n[{pattern}]")
            for line_no, line in matches:
                print(f"  {line_no}: {line}")

    if not found_any:
        print("No suspicious patterns found.")

# ------------------------------------------------------------------
# 2. FUNCTION DEFINITIONS
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("2. FUNCTION DEFINITIONS")
print("=" * 80)

for f in ["src/train_e2.py", "src/models/e2_concat_fusion.py"]:
    print(f"\n--- {f} ---")

    tree = ast.parse(Path(f).read_text(encoding="utf-8"))

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            args = [a.arg for a in node.args.args]
            print(f"  def {node.name}({', '.join(args)})  [line {node.lineno}]")

# ------------------------------------------------------------------
# 3. CLASS DEFINITIONS
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("3. CLASS DEFINITIONS")
print("=" * 80)

f = "src/models/e2_concat_fusion.py"
tree = ast.parse(Path(f).read_text(encoding="utf-8"))

for node in ast.walk(tree):
    if isinstance(node, ast.ClassDef):
        print(f"\nCLASS: {node.name} [line {node.lineno}]")

        for child in node.body:
            if isinstance(child, ast.FunctionDef):
                args = [a.arg for a in child.args.args]
                print(
                    f"  method {child.name}"
                    f"({', '.join(args)})"
                    f" [line {child.lineno}]"
                )

# ------------------------------------------------------------------
# 4. PRINT E2 MODEL CONSTRUCTOR + FORWARD METHODS
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("4. E2 MODEL CODE")
print("=" * 80)

text = Path("src/models/e2_concat_fusion.py").read_text(encoding="utf-8")
lines = text.splitlines()

keywords = [
    "class E2",
    "def __init__",
    "def forward",
    "def emoji_forward",
    "def text_forward",
    "def set_stocktwits_vocab",
]

for keyword in keywords:
    print(f"\n--- Searching: {keyword} ---")

    for i, line in enumerate(lines, 1):
        if keyword in line:
            start = max(0, i - 2)
            end = min(len(lines), i + 45)

            for j in range(start, end):
                print(f"{j+1:4}: {lines[j]}")

            print("-" * 60)

# ------------------------------------------------------------------
# 5. TRAIN_E2 FUNCTION
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("5. COMPLETE train_e2() FUNCTION")
print("=" * 80)

text = Path("src/train_e2.py").read_text(encoding="utf-8")
lines = text.splitlines()

start = None
end = None

for i, line in enumerate(lines):
    if line.startswith("def train_e2("):
        start = i

        # Find next top-level def
        for j in range(i + 1, len(lines)):
            if lines[j].startswith("def ") or lines[j].startswith("class "):
                end = j
                break

        if end is None:
            end = len(lines)

        break

if start is not None:
    for i in range(start, end):
        print(f"{i+1:4}: {lines[i]}")
else:
    print("❌ train_e2() not found")

# ------------------------------------------------------------------
# 6. MAIN FUNCTION
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("6. COMPLETE main() FUNCTION")
print("=" * 80)

start = None
end = None

for i, line in enumerate(lines):
    if line.startswith("def main("):
        start = i

        for j in range(i + 1, len(lines)):
            if lines[j].startswith("def ") or lines[j].startswith("class "):
                end = j
                break

        if end is None:
            end = len(lines)

        break

if start is not None:
    for i in range(start, end):
        print(f"{i+1:4}: {lines[i]}")
else:
    print("❌ main() not found")

# ------------------------------------------------------------------
# 7. STATIC NAME ANALYSIS
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("7. STATIC NAME ANALYSIS")
print("=" * 80)

try:
    import pyflakes.api
    import pyflakes.reporter
    import io

    for f in ["src/train_e2.py", "src/models/e2_concat_fusion.py"]:
        print(f"\n--- Pyflakes: {f} ---")

        output = io.StringIO()
        reporter = pyflakes.reporter.Reporter(output, output)

        status = pyflakes.api.checkPath(f, reporter)

        result = output.getvalue().strip()

        if result:
            print(result)
        else:
            print("✅ No Pyflakes issues.")

except ImportError:
    print("Pyflakes not installed. Installing...")
    !pip -q install pyflakes

    import pyflakes.api
    import pyflakes.reporter
    import io

    for f in ["src/train_e2.py", "src/models/e2_concat_fusion.py"]:
        print(f"\n--- Pyflakes: {f} ---")

        output = io.StringIO()
        reporter = pyflakes.reporter.Reporter(output, output)

        status = pyflakes.api.checkPath(f, reporter)

        result = output.getvalue().strip()

        if result:
            print(result)
        else:
            print("✅ No Pyflakes issues.")

# ------------------------------------------------------------------
# 8. EMBEDDING ARTIFACT
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("8. E2 EMBEDDING ARTIFACT")
print("=" * 80)

import torch

artifact = Path(
    "models/emoji_embeddings/"
    "stocktwits_emoji_embedding_e2_1610x32.pt"
)

print("Path:", artifact)

if artifact.exists():
    data = torch.load(artifact, map_location="cpu")

    print("✅ Artifact exists")
    print("Keys:", list(data.keys()))

    for k, v in data.items():
        if hasattr(v, "shape"):
            print(f"{k}: shape={v.shape}")
        elif isinstance(v, dict):
            print(f"{k}: dict with {len(v)} entries")
        else:
            print(f"{k}: {v}")

    emb = data.get("emoji_embedding")

    if emb is not None:
        print("\nEmbedding shape:", emb.shape)

        if tuple(emb.shape) == (1610, 32):
            print("✅ Correct: 1610 × 32")
        else:
            print("❌ WRONG SHAPE")

else:
    print("❌ Artifact missing")

# ------------------------------------------------------------------
# 9. FINAL CHECKLIST
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("9. FINAL CHECKLIST")
print("=" * 80)

train_text = Path("src/train_e2.py").read_text(encoding="utf-8")
model_text = Path("src/models/e2_concat_fusion.py").read_text(encoding="utf-8")

checks = {
    "E2 training function exists":
        "def train_e2" in train_text,

    "E2 model exists":
        "class E2" in model_text,

    "emoji_forward exists":
        "def emoji_forward" in model_text,

    "1610 vocabulary referenced":
        "1610" in train_text or "1610" in model_text,

    "32-dimensional emoji representation":
        "32" in train_text or "32" in model_text,

    "Old vocab[ reference exists":
        "vocab[" in train_text,

    "Old emoji_to_id inside train_e2":
        "emoji_to_id" in train_text,

    "Old 32-vocab hardcoding":
        "vocab_size=32" in train_text or "emoji_vocab_size=32" in train_text,

    "Old pretrained 32d artifact reference":
        "pretrained_emoji_32d.pt" in train_text or
        "pretrained_emoji_32d.pt" in model_text,

    "Guard still exists":
        "assert False" in train_text,
}

for name, value in checks.items():
    if "Old" in name or "Guard" in name:
        print(("❌ FOUND" if value else "✅ CLEAN"), "-", name)
    else:
        print(("✅ YES" if value else "❌ NO"), "-", name)

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

/content/SentimentAnalysis
                    E2 DEEP CODE AUDIT

FILE: src/train_e2.py
✅ Exists: 15,692 bytes

FILE: src/models/e2_concat_fusion.py
✅ Exists: 17,214 bytes

FILE: config.yaml
✅ Exists: 807 bytes

FILE: requirements.txt
✅ Exists: 179 bytes

1. STALE / SUSPICIOUS REFERENCES

--- src/train_e2.py ---

[vocab[]
  242: print(f"Emoji vocab size (train-only): {vocab['vocab_size']}")
  355: e_ids, e_mask = encode_emoji_list(emojis, vocab["emoji_to_id"], max_emojis)

[vocab.]
  243: save_vocab_path = os.path.join("models/emoji_embeddings", "emoji_vocab.json")

[emoji_to_id]
  200: "emoji_vocab_size": len(emoji_to_id),  # Use actual StockTwits vocabulary size
  355: e_ids, e_mask = encode_emoji_list(emojis, vocab["emoji_to_id"], max_emojis)

[emoji_vocab]
  45: build_emoji_vocab,
  200: "emoji_vocab_size": len(emoji_to_id),  # Use actual StockTwits vocabulary size
  241: vocab = build_emoji_vocab(train_df["emoji_list"].tolist())
  243: save_vocab_path = os.path.join("models/emoji

In [58]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("src/models/e2_concat_fusion.py")

new_model = r'''
import os
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel


class E2ConcatFusionModel(nn.Module):
    """
    E2: Text + pretrained emoji embedding with concatenation fusion.

    Text branch:
        frozen BERT -> 768-d representation

    Emoji branch:
        frozen StockTwits emoji embedding -> mean pooling -> 32-d representation

    Fusion:
        concatenate text and emoji representations -> 800-d

    Classifier:
        800 -> 256 -> 3
    """

    def __init__(
        self,
        model_name: str = "bert-base-uncased",
        text_dim: int = 768,
        emoji_dim: int = 32,
        num_labels: int = 3,
        dropout: float = 0.3,
        classifier_hidden: int = 256,
        freeze_encoder: bool = True,
        emoji_vocab_size: int = 1610,
        max_emojis: int = 8,
        pretrained_emoji_path: str = "",
    ) -> None:

        super().__init__()

        self.model_name = model_name
        self.text_dim = text_dim
        self.emoji_dim = emoji_dim
        self.emoji_vocab_size = emoji_vocab_size
        self.max_emojis = max_emojis

        # ----------------------------------------------------------
        # Frozen BERT text encoder
        # ----------------------------------------------------------
        config = AutoConfig.from_pretrained(
            model_name,
            num_labels=num_labels,
        )

        self.encoder = AutoModel.from_pretrained(
            model_name,
            config=config,
        )

        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False

        # ----------------------------------------------------------
        # Emoji embedding
        # ----------------------------------------------------------
        self.emoji_encoder = nn.Embedding(
            emoji_vocab_size,
            emoji_dim,
            padding_idx=0,
        )

        # Random initialization first.
        nn.init.normal_(
            self.emoji_encoder.weight,
            mean=0.0,
            std=0.1,
        )

        # Padding / UNK row is kept deterministic.
        with torch.no_grad():
            self.emoji_encoder.weight[0].zero_()

        # ----------------------------------------------------------
        # Optionally load a final embedding artifact.
        # In the current E2 pipeline the artifact is loaded
        # explicitly in train_e2.py, so this is normally empty.
        # ----------------------------------------------------------
        if pretrained_emoji_path:
            if not os.path.exists(pretrained_emoji_path):
                raise FileNotFoundError(
                    f"Emoji embedding not found: {pretrained_emoji_path}"
                )

            artifact = torch.load(
                pretrained_emoji_path,
                map_location="cpu",
            )

            embedding = artifact["emoji_embedding"]

            if tuple(embedding.shape) != (
                emoji_vocab_size,
                emoji_dim,
            ):
                raise ValueError(
                    "Emoji embedding shape mismatch: "
                    f"expected {(emoji_vocab_size, emoji_dim)}, "
                    f"got {tuple(embedding.shape)}"
                )

            with torch.no_grad():
                self.emoji_encoder.weight.copy_(embedding)

        # E2 uses the final transferred embedding as frozen.
        for param in self.emoji_encoder.parameters():
            param.requires_grad = False

        # ----------------------------------------------------------
        # Fusion classifier
        # ----------------------------------------------------------
        fused_dim = text_dim + emoji_dim

        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, classifier_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(classifier_hidden, num_labels),
        )

    # --------------------------------------------------------------
    # Text representation
    # --------------------------------------------------------------
    def text_forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
    ) -> torch.Tensor:

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        last_hidden = outputs.last_hidden_state

        mask = attention_mask.unsqueeze(-1).float()

        summed = (last_hidden * mask).sum(dim=1)

        counts = mask.sum(dim=1).clamp(min=1e-9)

        return summed / counts

    # --------------------------------------------------------------
    # Emoji representation
    # --------------------------------------------------------------
    def emoji_forward(
        self,
        emoji_ids: torch.Tensor,
        emoji_masks: torch.Tensor,
    ) -> torch.Tensor:
        """
        Mean-pool the frozen emoji embeddings.

        emoji_ids:
            [B, max_emojis]

        emoji_masks:
            [B, max_emojis]

        returns:
            [B, emoji_dim]
        """

        embeddings = self.emoji_encoder(
            emoji_ids
        )

        mask = emoji_masks.unsqueeze(-1).float()

        summed = (embeddings * mask).sum(dim=1)

        counts = mask.sum(dim=1).clamp(min=1e-9)

        return summed / counts

    # --------------------------------------------------------------
    # Forward using precomputed text embeddings
    # --------------------------------------------------------------
    def forward_fused(
        self,
        text_emb: torch.Tensor,
        emoji_ids: torch.Tensor,
        emoji_masks: torch.Tensor,
    ) -> torch.Tensor:

        emoji_rep = self.emoji_forward(
            emoji_ids,
            emoji_masks,
        )

        fused = torch.cat(
            [text_emb, emoji_rep],
            dim=-1,
        )

        return self.classifier(fused)

    # --------------------------------------------------------------
    # Standard forward for completeness
    # --------------------------------------------------------------
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        emoji_ids: torch.Tensor,
        emoji_masks: torch.Tensor,
    ) -> torch.Tensor:

        text_rep = self.text_forward(
            input_ids,
            attention_mask,
        )

        return self.forward_fused(
            text_rep,
            emoji_ids,
            emoji_masks,
        )
'''

path.write_text(new_model.strip() + "\n", encoding="utf-8")

print("✅ Replaced e2_concat_fusion.py with clean E2 model.")

/content/SentimentAnalysis
✅ Replaced e2_concat_fusion.py with clean E2 model.


In [59]:
%cd /content/SentimentAnalysis

from pathlib import Path

lines = Path("src/train_e2.py").read_text(encoding="utf-8").splitlines()

print("===== TRAIN_E2 IMPORTS =====")
for i, line in enumerate(lines[:60], 1):
    print(f"{i:3}: {line}")

print("\n===== CONFIG.YAML =====")
print(Path("config.yaml").read_text(encoding="utf-8"))

/content/SentimentAnalysis
===== TRAIN_E2 IMPORTS =====
  1: """E2 — text + pretrained emoji embedding, concatenation fusion.
  2: 
  3: MIRRORS E0/E1 exactly (seed, class weights, optimizer, batch, max_length, metrics,
  4: model-selection criterion) except it uses PRETRAINED emoji embeddings (frozen)
  5: learned from the TweetEval emoji prediction task, fused by concatenation.
  6: 
  7: Controlled-experiment contract:
  8: - Same train/validation/test examples and labels as E0/E1.
  9: - Text branch receives ONLY ``text_without_emoji``.
 10: - Emoji branch receives ONLY ``emoji_list`` (pretrained frozen embedding).
 11: - ``original_text`` is never passed to BERT.
 12: - Emoji embeddings are pretrained on TweetEval emoji prediction task (20 classes)
 13:   and frozen during E2 training.
 14: 
 15: This script is PREPARED but must NOT be trained until approved.
 16: """
 17: 
 18: from __future__ import annotations
 19: 
 20: import hashlib
 21: import json
 22: import os
 23: impor

In [60]:
%cd /content/SentimentAnalysis

from pathlib import Path

train_script = r'''
"""E2 — Text + pretrained emoji embedding with concatenation fusion.

Controlled experiment:
- Same canonical StockTwits train/validation/test splits as E0/E1.
- Text branch receives ONLY text_without_emoji.
- Emoji branch receives ONLY emoji_list.
- BERT is frozen.
- Final StockTwits emoji embedding is frozen.
- Emoji representation is mean-pooled.
- Text representation is the same frozen-BERT mean pooling used by E0/E1.
- Fusion is concatenation: 768 + 32 = 800.
- Classifier: 800 -> 256 -> 3.
- Best checkpoint selected using validation Macro-F1.
"""

from __future__ import annotations

import hashlib
import json
import os
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

from src.models.e2_concat_fusion import E2ConcatFusionModel
from src.embeddings.emoji_encoder import (
    build_emoji_vocab,
    encode_emoji_list,
)
from src.utils.seed import set_seed


LABEL_NAMES = {
    0: "Bearish",
    1: "Neutral",
    2: "Bullish",
}


# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

def load_config(path: str = "config.yaml") -> dict:
    import yaml

    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


# ---------------------------------------------------------------------------
# Text featurization
# ---------------------------------------------------------------------------

@torch.no_grad()
def featurize_texts(
    texts,
    tokenizer,
    encoder,
    max_length,
    batch_size,
    device,
):
    """
    Convert text_without_emoji into frozen BERT mean-pooled embeddings.
    Returns numpy array of shape [N, 768].
    """

    encoder.eval()

    all_embeddings = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Featurizing text",
    ):
        batch_texts = texts[start:start + batch_size]

        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )

        input_ids = encoded["input_ids"].to(device)
        attention_mask = encoded["attention_mask"].to(device)

        outputs = encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )

        hidden = outputs.last_hidden_state

        mask = attention_mask.unsqueeze(-1).float()

        summed = (hidden * mask).sum(dim=1)

        counts = mask.sum(dim=1).clamp(min=1e-9)

        pooled = summed / counts

        all_embeddings.append(
            pooled.cpu().numpy()
        )

    return np.concatenate(all_embeddings, axis=0)


# ---------------------------------------------------------------------------
# Dataset
# ---------------------------------------------------------------------------

class EmojiDataset(Dataset):

    def __init__(
        self,
        text_embeds,
        emoji_ids,
        emoji_masks,
        labels,
    ):
        self.text_embeds = torch.tensor(
            text_embeds,
            dtype=torch.float32,
        )

        self.emoji_ids = torch.tensor(
            emoji_ids,
            dtype=torch.long,
        )

        self.emoji_masks = torch.tensor(
            emoji_masks,
            dtype=torch.long,
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long,
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return (
            self.text_embeds[idx],
            self.emoji_ids[idx],
            self.emoji_masks[idx],
            self.labels[idx],
        )


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def train_e2(
    model,
    train_ds,
    val_ds,
    cfg,
    device,
    class_weights,
    out_dir,
):
    train_cfg = cfg["training"]

    batch_size = train_cfg["batch_size"]
    epochs = train_cfg["epochs"]
    lr = train_cfg["learning_rate"]
    wd = train_cfg.get("weight_decay", 1e-4)

    # Preserve effective batch size of 32.
    # Use physical batch size 8 for compatibility with the original
    # RTX-3050-oriented implementation.
    physical_batch_size = 8
    accumulation_steps = batch_size // physical_batch_size

    train_loader = DataLoader(
        train_ds,
        batch_size=physical_batch_size,
        shuffle=True,
        num_workers=0,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=physical_batch_size,
        shuffle=False,
        num_workers=0,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device)
    )

    # BERT and emoji embedding are frozen.
    # Only the classifier is trainable.
    trainable = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    optimizer = torch.optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=wd,
    )

    use_amp = (
        train_cfg.get("mixed_precision", True)
        and device.type == "cuda"
    )

    if use_amp:
        from torch.cuda.amp import autocast, GradScaler

        scaler = GradScaler()
    else:
        autocast = None
        scaler = None

    history = {
        "train_loss": [],
        "val_loss": [],
        "val_acc": [],
        "val_macro_f1": [],
    }

    best_epoch = -1
    best_val_f1 = -1.0

    patience = train_cfg.get(
        "early_stop_patience",
        5,
    )

    no_improve = 0

    os.makedirs(out_dir, exist_ok=True)

    model.to(device)

    print(
        f"\nTrainable parameters: "
        f"{sum(p.numel() for p in trainable):,}"
    )

    for epoch in range(1, epochs + 1):

        # ---------------------------------------------------------------
        # Training
        # ---------------------------------------------------------------

        model.train()

        # Keep frozen branches in evaluation mode.
        model.encoder.eval()
        model.emoji_encoder.eval()

        total_loss = 0.0
        n_batches = 0

        optimizer.zero_grad()

        for i, (
            text_emb,
            e_ids,
            e_mask,
            labels,
        ) in enumerate(train_loader):

            text_emb = text_emb.to(device)
            e_ids = e_ids.to(device)
            e_mask = e_mask.to(device)
            labels = labels.to(device)

            if use_amp:

                with autocast():

                    emoji_rep = model.emoji_forward(
                        e_ids,
                        e_mask,
                    )

                    fused = torch.cat(
                        [text_emb, emoji_rep],
                        dim=-1,
                    )

                    logits = model.classifier(fused)

                    loss = criterion(
                        logits,
                        labels,
                    )

                    loss_for_backward = (
                        loss / accumulation_steps
                    )

            else:

                emoji_rep = model.emoji_forward(
                    e_ids,
                    e_mask,
                )

                fused = torch.cat(
                    [text_emb, emoji_rep],
                    dim=-1,
                )

                logits = model.classifier(fused)

                loss = criterion(
                    logits,
                    labels,
                )

                loss_for_backward = (
                    loss / accumulation_steps
                )

            if use_amp:
                scaler.scale(
                    loss_for_backward
                ).backward()
            else:
                loss_for_backward.backward()

            if (
                (i + 1) % accumulation_steps == 0
                or (i + 1) == len(train_loader)
            ):

                if use_amp:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                optimizer.zero_grad()

            total_loss += loss.item()
            n_batches += 1

        train_loss = (
            total_loss / max(n_batches, 1)
        )

        # ---------------------------------------------------------------
        # Validation
        # ---------------------------------------------------------------

        model.eval()

        val_loss = 0.0
        all_preds = []
        all_true = []

        with torch.no_grad():

            for (
                text_emb,
                e_ids,
                e_mask,
                labels,
            ) in val_loader:

                text_emb = text_emb.to(device)
                e_ids = e_ids.to(device)
                e_mask = e_mask.to(device)
                labels = labels.to(device)

                if use_amp:

                    with autocast():

                        emoji_rep = model.emoji_forward(
                            e_ids,
                            e_mask,
                        )

                        fused = torch.cat(
                            [text_emb, emoji_rep],
                            dim=-1,
                        )

                        logits = model.classifier(
                            fused
                        )

                        loss = criterion(
                            logits,
                            labels,
                        )

                else:

                    emoji_rep = model.emoji_forward(
                        e_ids,
                        e_mask,
                    )

                    fused = torch.cat(
                        [text_emb, emoji_rep],
                        dim=-1,
                    )

                    logits = model.classifier(
                        fused
                    )

                    loss = criterion(
                        logits,
                        labels,
                    )

                val_loss += loss.item()

                preds = logits.argmax(
                    dim=-1
                )

                all_preds.extend(
                    preds.cpu().numpy().tolist()
                )

                all_true.extend(
                    labels.cpu().numpy().tolist()
                )

        val_loss /= max(
            len(val_loader),
            1,
        )

        val_acc = accuracy_score(
            all_true,
            all_preds,
        )

        val_macro_f1 = f1_score(
            all_true,
            all_preds,
            average="macro",
            zero_division=0,
        )

        history["train_loss"].append(
            train_loss
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_acc"].append(
            val_acc
        )

        history["val_macro_f1"].append(
            val_macro_f1
        )

        print(
            f"Epoch {epoch}/{epochs} | "
            f"TrainLoss {train_loss:.4f} | "
            f"ValLoss {val_loss:.4f} | "
            f"ValAcc {val_acc:.4f} | "
            f"ValMacroF1 {val_macro_f1:.4f}"
        )

        # ---------------------------------------------------------------
        # Best checkpoint
        # ---------------------------------------------------------------

        if val_macro_f1 > best_val_f1:

            best_val_f1 = val_macro_f1
            best_epoch = epoch
            no_improve = 0

            checkpoint = {
                "classifier_state": (
                    model.classifier.state_dict()
                ),
                "emoji_encoder_state": (
                    model.emoji_encoder.state_dict()
                ),
                "text_dim": model.text_dim,
                "emoji_dim": model.emoji_dim,
                "emoji_vocab_size": (
                    model.emoji_vocab_size
                ),
                "best_val_f1": best_val_f1,
                "best_epoch": best_epoch,
                "class_weights": (
                    class_weights.cpu().tolist()
                ),
                "seed": cfg["project"]["seed"],
                "fusion": "concatenation",
                "text_input": "text_without_emoji",
                "emoji_input": "emoji_list",
            }

            torch.save(
                checkpoint,
                os.path.join(
                    out_dir,
                    "best_model.pt",
                ),
            )

            print(
                f"  ✓ New best checkpoint "
                f"(Val Macro-F1={best_val_f1:.4f})"
            )

        else:

            no_improve += 1

            if no_improve >= patience:

                print(
                    f"Early stopping at epoch {epoch}"
                )

                break

    # Save training history
    with open(
        os.path.join(
            out_dir,
            "training_history.json",
        ),
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            {
                "best_epoch": best_epoch,
                "best_val_macro_f1": best_val_f1,
                "history": history,
            },
            f,
            indent=2,
        )

    return {
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_val_f1,
        "history": history,
    }


# ---------------------------------------------------------------------------
# Test evaluation
# ---------------------------------------------------------------------------

@torch.no_grad()
def evaluate_e2(
    model,
    test_ds,
    cfg,
    device,
    class_weights,
    checkpoint_path,
    out_dir,
):
    """
    Load best classifier checkpoint and evaluate the untouched
    canonical test split.
    """

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
    )

    model.classifier.load_state_dict(
        checkpoint["classifier_state"]
    )

    if "emoji_encoder_state" in checkpoint:
        model.emoji_encoder.load_state_dict(
            checkpoint["emoji_encoder_state"]
        )

    model.to(device)
    model.eval()

    test_loader = DataLoader(
        test_ds,
        batch_size=8,
        shuffle=False,
        num_workers=0,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights.to(device)
    )

    all_preds = []
    all_true = []
    all_probs = []

    test_loss = 0.0

    for (
        text_emb,
        e_ids,
        e_mask,
        labels,
    ) in test_loader:

        text_emb = text_emb.to(device)
        e_ids = e_ids.to(device)
        e_mask = e_mask.to(device)
        labels = labels.to(device)

        emoji_rep = model.emoji_forward(
            e_ids,
            e_mask,
        )

        fused = torch.cat(
            [text_emb, emoji_rep],
            dim=-1,
        )

        logits = model.classifier(fused)

        loss = criterion(
            logits,
            labels,
        )

        test_loss += loss.item()

        probs = torch.softmax(
            logits,
            dim=-1,
        )

        preds = logits.argmax(
            dim=-1
        )

        all_probs.extend(
            probs.cpu().numpy().tolist()
        )

        all_preds.extend(
            preds.cpu().numpy().tolist()
        )

        all_true.extend(
            labels.cpu().numpy().tolist()
        )

    test_loss /= max(
        len(test_loader),
        1,
    )

    accuracy = accuracy_score(
        all_true,
        all_preds,
    )

    macro_precision = precision_score(
        all_true,
        all_preds,
        average="macro",
        zero_division=0,
    )

    macro_recall = recall_score(
        all_true,
        all_preds,
        average="macro",
        zero_division=0,
    )

    macro_f1 = f1_score(
        all_true,
        all_preds,
        average="macro",
        zero_division=0,
    )

    report = classification_report(
        all_true,
        all_preds,
        labels=[0, 1, 2],
        target_names=[
            LABEL_NAMES[0],
            LABEL_NAMES[1],
            LABEL_NAMES[2],
        ],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(
        all_true,
        all_preds,
        labels=[0, 1, 2],
    )

    # ---------------------------------------------------------------
    # Save confusion matrix
    # ---------------------------------------------------------------

    cm_df = pd.DataFrame(
        cm,
        index=[
            "Actual_Bearish",
            "Actual_Neutral",
            "Actual_Bullish",
        ],
        columns=[
            "Pred_Bearish",
            "Pred_Neutral",
            "Pred_Bullish",
        ],
    )

    cm_df.to_csv(
        os.path.join(
            out_dir,
            "confusion_matrix.csv",
        )
    )

    # ---------------------------------------------------------------
    # Save predictions
    # ---------------------------------------------------------------

    predictions_df = pd.DataFrame(
        {
            "true_label": all_true,
            "predicted_label": all_preds,
            "true_class": [
                LABEL_NAMES[x]
                for x in all_true
            ],
            "predicted_class": [
                LABEL_NAMES[x]
                for x in all_preds
            ],
            "prob_bearish": [
                p[0] for p in all_probs
            ],
            "prob_neutral": [
                p[1] for p in all_probs
            ],
            "prob_bullish": [
                p[2] for p in all_probs
            ],
        }
    )

    predictions_df.to_csv(
        os.path.join(
            out_dir,
            "predictions.csv",
        ),
        index=False,
    )

    # ---------------------------------------------------------------
    # Save metrics
    # ---------------------------------------------------------------

    metrics = {
        "experiment": "E2",
        "description": (
            "Text + partially pretrained emoji embeddings "
            "with concatenation fusion"
        ),
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "test_loss": test_loss,
        "best_epoch": checkpoint.get(
            "best_epoch",
            -1,
        ),
        "best_val_macro_f1": checkpoint.get(
            "best_val_f1",
            -1,
        ),
        "seed": cfg["project"]["seed"],
        "text_input": "text_without_emoji",
        "emoji_input": "emoji_list",
        "text_dim": model.text_dim,
        "emoji_dim": model.emoji_dim,
        "emoji_vocab_size": model.emoji_vocab_size,
        "fusion": "concatenation",
        "embedding_source": "TweetEval overlap + deterministic random remainder",
        "tweet_eval_pretrained_count": 19,
        "random_initialized_count": 1591,
        "report": report,
    }

    with open(
        os.path.join(
            out_dir,
            "metrics.json",
        ),
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2,
        )

    # ---------------------------------------------------------------
    # Print final results
    # ---------------------------------------------------------------

    print("\n" + "=" * 70)
    print("E2 OFFICIAL TEST RESULTS")
    print("=" * 70)

    print(
        f"Accuracy          : {accuracy:.6f}"
    )

    print(
        f"Macro Precision   : {macro_precision:.6f}"
    )

    print(
        f"Macro Recall      : {macro_recall:.6f}"
    )

    print(
        f"Macro F1          : {macro_f1:.6f}"
    )

    print(
        f"Best Epoch        : {checkpoint.get('best_epoch', -1)}"
    )

    print(
        f"Best Val Macro-F1 : {checkpoint.get('best_val_f1', -1):.6f}"
    )

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            all_true,
            all_preds,
            labels=[0, 1, 2],
            target_names=[
                LABEL_NAMES[0],
                LABEL_NAMES[1],
                LABEL_NAMES[2],
            ],
            zero_division=0,
        )
    )

    return metrics


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():

    cfg = load_config()

    seed = cfg["project"]["seed"]

    set_seed(seed)

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(
        f"Using device: {device}"
    )

    # ---------------------------------------------------------------
    # Configuration
    # ---------------------------------------------------------------

    text_cfg = cfg["text_model"]
    model_name = text_cfg["name"]
    max_length = text_cfg["max_length"]

    train_cfg = cfg["training"]

    emoji_dim = cfg["emoji"]["embedding_dim"]

    max_emojis = 8

    out_dir = "results/E2"

    os.makedirs(
        out_dir,
        exist_ok=True,
    )

    # ---------------------------------------------------------------
    # Load canonical splits
    # ---------------------------------------------------------------

    train_df = pd.read_json(
        "data/processed/canonical/final_train.jsonl",
        lines=True,
    )

    val_df = pd.read_json(
        "data/processed/canonical/final_validation.jsonl",
        lines=True,
    )

    test_df = pd.read_json(
        "data/processed/canonical/final_test.jsonl",
        lines=True,
    )

    print(
        f"Train: {len(train_df)}"
    )

    print(
        f"Validation: {len(val_df)}"
    )

    print(
        f"Test: {len(test_df)}"
    )

    # ---------------------------------------------------------------
    # Train-only emoji vocabulary
    # ---------------------------------------------------------------

    vocab = build_emoji_vocab(
        train_df["emoji_list"].tolist()
    )

    emoji_to_id = vocab["emoji_to_id"]

    print(
        f"Emoji vocab size (train-only): "
        f"{vocab['vocab_size']}"
    )

    assert vocab["vocab_size"] == 1610

    # Save vocabulary for reproducibility
    from src.embeddings.emoji_encoder import save_vocab

    save_vocab(
        vocab,
        "models/emoji_embeddings/emoji_vocab.json",
    )

    # ---------------------------------------------------------------
    # Class weights
    # ---------------------------------------------------------------

    counts = Counter(
        train_df["label"]
    )

    total = sum(
        counts.values()
    )

    n_classes = 3

    class_weights = torch.tensor(
        [
            total / (
                n_classes * counts[c]
            )
            for c in [0, 1, 2]
        ],
        dtype=torch.float32,
    )

    print(
        "Class weights (train-only):",
        class_weights.tolist(),
    )

    # ---------------------------------------------------------------
    # Final 1610 x 32 emoji embedding
    # ---------------------------------------------------------------

    embedding_path = (
        "models/emoji_embeddings/"
        "stocktwits_emoji_embedding_e2_1610x32.pt"
    )

    if not os.path.exists(
        embedding_path
    ):
        raise FileNotFoundError(
            f"Missing E2 embedding artifact: "
            f"{embedding_path}"
        )

    embedding_artifact = torch.load(
        embedding_path,
        map_location="cpu",
    )

    embedding = embedding_artifact[
        "emoji_embedding"
    ]

    assert embedding.shape == (
        1610,
        32,
    )

    assert embedding_artifact[
        "vocab_size"
    ] == 1610

    assert embedding_artifact[
        "embedding_dim"
    ] == 32

    print(
        f"Loaded E2 emoji embedding: "
        f"{tuple(embedding.shape)}"
    )

    print(
        f"TweetEval pretrained rows: "
        f"{embedding_artifact.get('tweet_eval_pretrained_count', 19)}"
    )

    print(
        f"Random rows: "
        f"{embedding_artifact.get('random_initialized_count', 1591)}"
    )

    # ---------------------------------------------------------------
    # Tokenizer + E2 model
    # ---------------------------------------------------------------

    tokenizer = AutoTokenizer.from_pretrained(
        model_name
    )

    model = E2ConcatFusionModel(
        model_name=model_name,
        text_dim=768,
        emoji_dim=32,
        num_labels=3,
        dropout=cfg["classifier_head"]["dropout"],
        classifier_hidden=cfg[
            "classifier_head"
        ]["classifier_hidden"],
        freeze_encoder=True,
        emoji_vocab_size=1610,
        max_emojis=max_emojis,
        pretrained_emoji_path="",
    )

    # Load final transferred embedding
    with torch.no_grad():
        model.emoji_encoder.weight.copy_(
            embedding
        )

    # Keep emoji embeddings frozen
    for param in model.emoji_encoder.parameters():
        param.requires_grad = False

    # Keep BERT frozen
    for param in model.encoder.parameters():
        param.requires_grad = False

    print(
        "Loaded final E2 emoji embedding "
        "into model."
    )

    # ---------------------------------------------------------------
    # Frozen-BERT text embeddings
    # ---------------------------------------------------------------

    encoder = model.encoder.to(device)

    hash_str = hashlib.sha1(
        (
            train_df[
                "text_without_emoji"
            ].sum()
            +
            val_df[
                "text_without_emoji"
            ].sum()
            +
            test_df[
                "text_without_emoji"
            ].sum()
        ).encode("utf-8")
    ).hexdigest()

    cache_dir = (
        "results/E2/embeddings_cache"
    )

    os.makedirs(
        cache_dir,
        exist_ok=True,
    )

    def get_emb(
        split_df,
        name,
    ):

        cache_path = os.path.join(
            cache_dir,
            f"text_{name}_{hash_str[:8]}.npy",
        )

        if os.path.exists(
            cache_path
        ):

            print(
                f"Using cached text embeddings: "
                f"{cache_path}"
            )

            return np.load(
                cache_path
            )

        embeddings = featurize_texts(
            split_df[
                "text_without_emoji"
            ].tolist(),
            tokenizer,
            encoder,
            max_length,
            train_cfg["batch_size"],
            device,
        )

        np.save(
            cache_path,
            embeddings,
        )

        return embeddings

    X_train = get_emb(
        train_df,
        "train",
    )

    X_val = get_emb(
        val_df,
        "validation",
    )

    X_test = get_emb(
        test_df,
        "test",
    )

    # ---------------------------------------------------------------
    # Encode emoji lists
    # ---------------------------------------------------------------

    def encode_split(df):

        ids = []
        masks = []

        for emojis in df[
            "emoji_list"
        ].tolist():

            e_ids, e_mask = encode_emoji_list(
                emojis,
                emoji_to_id,
                max_emojis,
            )

            ids.append(e_ids)
            masks.append(e_mask)

        return (
            np.asarray(ids),
            np.asarray(masks),
        )

    e_train_ids, e_train_mask = encode_split(
        train_df
    )

    e_val_ids, e_val_mask = encode_split(
        val_df
    )

    e_test_ids, e_test_mask = encode_split(
        test_df
    )

    # ---------------------------------------------------------------
    # Dataset objects
    # ---------------------------------------------------------------

    train_ds = EmojiDataset(
        X_train,
        e_train_ids,
        e_train_mask,
        train_df["label"].to_numpy(),
    )

    val_ds = EmojiDataset(
        X_val,
        e_val_ids,
        e_val_mask,
        val_df["label"].to_numpy(),
    )

    test_ds = EmojiDataset(
        X_test,
        e_test_ids,
        e_test_mask,
        test_df["label"].to_numpy(),
    )

    # ---------------------------------------------------------------
    # Train
    # ---------------------------------------------------------------

    print(
        "\n[E2] Starting training..."
    )

    training_result = train_e2(
        model=model,
        train_ds=train_ds,
        val_ds=val_ds,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
        out_dir=out_dir,
    )

    print(
        "\n[E2] Training complete."
    )

    # ---------------------------------------------------------------
    # Test evaluation
    # ---------------------------------------------------------------

    checkpoint_path = os.path.join(
        out_dir,
        "best_model.pt",
    )

    if not os.path.exists(
        checkpoint_path
    ):
        raise FileNotFoundError(
            "Best checkpoint was not created."
        )

    metrics = evaluate_e2(
        model=model,
        test_ds=test_ds,
        cfg=cfg,
        device=device,
        class_weights=class_weights,
        checkpoint_path=checkpoint_path,
        out_dir=out_dir,
    )

    print(
        "\n[E2] Complete."
    )

    return {
        "training": training_result,
        "test": metrics,
    }


if __name__ == "__main__":

    if os.environ.get("RUN_E2") == "1":
        main()

    else:
        print(
            "E2 is disabled. "
            "Set RUN_E2=1 to train."
        )
'''

Path("src/train_e2.py").write_text(
    train_script.strip() + "\n",
    encoding="utf-8",
)

print("✅ Completely replaced src/train_e2.py")
print("   Old stale train_e2 implementation removed.")

/content/SentimentAnalysis
✅ Completely replaced src/train_e2.py
   Old stale train_e2 implementation removed.


In [61]:
%cd /content/SentimentAnalysis

!python -m py_compile src/train_e2.py src/models/e2_concat_fusion.py

print("✅ Python syntax check passed.")

/content/SentimentAnalysis
✅ Python syntax check passed.


In [62]:
%cd /content/SentimentAnalysis

!pyflakes src/train_e2.py src/models/e2_concat_fusion.py

/content/SentimentAnalysis
src/train_e2.py:934:5: local variable 'emoji_dim' is assigned to but never used


In [63]:
%cd /content/SentimentAnalysis

from pathlib import Path

path = Path("config.yaml")
text = path.read_text(encoding="utf-8")

text = text.replace(
    "fusion:\n",
    "fusion:\n  type: concatenation\n"
)

path.write_text(text, encoding="utf-8")

print("✅ config.yaml fusion setting fixed.")

/content/SentimentAnalysis
✅ config.yaml fusion setting fixed.


In [64]:
%cd /content/SentimentAnalysis
!RUN_E2=1 python -m src.train_e2

/content/SentimentAnalysis
Using device: cuda
Train: 91121
Validation: 20676
Test: 11966
Emoji vocab size (train-only): 1610
Class weights (train-only): [4.1664838790893555, 1.1576653718948364, 0.5273755192756653]
Loaded E2 emoji embedding: (1610, 32)
TweetEval pretrained rows: 19
Random rows: 1591
Loading weights: 100% 199/199 [00:00<00:00, 4387.46it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	ca

In [65]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [66]:
import os
import shutil

PROJECT = "/content/SentimentAnalysis"
DRIVE_ROOT = "/content/drive/MyDrive/SentimentAnalysis"

os.makedirs(DRIVE_ROOT, exist_ok=True)

# Copy E2 results
shutil.copytree(
    f"{PROJECT}/results/E2",
    f"{DRIVE_ROOT}/results/E2",
    dirs_exist_ok=True
)

# Copy E2 embedding artifact
os.makedirs(f"{DRIVE_ROOT}/models/emoji_embeddings", exist_ok=True)

shutil.copy2(
    f"{PROJECT}/models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt",
    f"{DRIVE_ROOT}/models/emoji_embeddings/stocktwits_emoji_embedding_e2_1610x32.pt"
)

print("E2 artifacts copied to Google Drive.")

E2 artifacts copied to Google Drive.


In [67]:
!find /content/drive/MyDrive/SentimentAnalysis/results/E2 -maxdepth 2 -type f | sort
!ls -lh /content/drive/MyDrive/SentimentAnalysis/models/emoji_embeddings/


/content/drive/MyDrive/SentimentAnalysis/results/E2/best_model.pt
/content/drive/MyDrive/SentimentAnalysis/results/E2/confusion_matrix.csv
/content/drive/MyDrive/SentimentAnalysis/results/E2/embeddings_cache/text_test_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E2/embeddings_cache/text_train_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E2/embeddings_cache/text_validation_608685ef.npy
/content/drive/MyDrive/SentimentAnalysis/results/E2/metrics.json
/content/drive/MyDrive/SentimentAnalysis/results/E2/predictions.csv
/content/drive/MyDrive/SentimentAnalysis/results/E2/training_history.json
total 233K
-rw------- 1 root root 233K Sep  7 22:07 stocktwits_emoji_embedding_e2_1610x32.pt
